In [5]:
import re
import csv
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# 0) Chemins du projet
# ============================================================
CODE_DIR = Path().resolve()
PROJECT_ROOT = CODE_DIR.parent.parent
DATA_DIR = PROJECT_ROOT / "DATA"

EXCEL_DIR = DATA_DIR / "Excels_code"

ROOT_VICON = DATA_DIR / "VICON_CSV"
ROOT_MP = DATA_DIR / "video"   # adapte en "Vidéo" si besoin

FPS_VICON = 100.0
FPS_MP = 30.0

PLANE_2D = "YZ"

OUT_PATH = EXCEL_DIR / f"ANGLE_elbow_MP_vs_VICON_plane{PLANE_2D}_noInterp.xlsx"

EXCEL_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_DIR =", DATA_DIR)
print("ROOT_VICON =", ROOT_VICON, ROOT_VICON.exists())
print("ROOT_MP =", ROOT_MP, ROOT_MP.exists())
print("Nb CSV Vicon =", len(list(ROOT_VICON.rglob("*.csv"))))
print("Nb pose MediaPipe =", len(list(ROOT_MP.rglob("*_pose.xlsx"))))


# ============================================================
# 1) Lecture CSV Vicon
# ============================================================
def read_vicon_csv(csv_path: Path) -> pd.DataFrame:
    with open(csv_path, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f)
        header_lines = [next(reader) for _ in range(5)]

    marker_row = header_lines[2]
    axis_row = header_lines[3]

    n = max(len(marker_row), len(axis_row))
    marker_row += [""] * (n - len(marker_row))
    axis_row += [""] * (n - len(axis_row))

    filled = []
    last = ""

    for m in marker_row:
        m = (m or "").strip()
        if m == "":
            filled.append(last)
        else:
            last = m
            filled.append(last)

    colnames = []

    for m, a in zip(filled, axis_row):
        m = (m or "").strip()
        a = (a or "").strip()

        if a in ["Frame", "Sub Frame"]:
            colnames.append(a)
        elif a in ["X", "Y", "Z"]:
            colnames.append(f"{m}_{a}")
        else:
            colnames.append(m if m else a)

    df = pd.read_csv(csv_path, skiprows=5, header=None, names=colnames, engine="python")
    df = df.dropna(axis=1, how="all")
    return df


def find_xyz_cols(cols, token):
    pat = re.compile(rf"(?:^|:)\s*{re.escape(token)}_([XYZ])\b", re.IGNORECASE)

    found = {}

    for c in cols:
        c2 = str(c).replace(" ", "")
        m = pat.search(c2)

        if m:
            axis = m.group(1).upper()
            if axis not in found or len(str(c)) < len(str(found[axis])):
                found[axis] = c

    return found.get("X"), found.get("Y"), found.get("Z")


# ============================================================
# 2) Matching fichiers
# ============================================================
def parse_vicon_name(csv_name: str):
    s = csv_name.upper().replace(".CSV", "")

    if s.startswith("SEATED"):
        cond = "SEATED"
    elif s.startswith("STANDING"):
        cond = "STANDING"
    elif s.startswith("SEMI"):
        cond = "SEMI"
    else:
        cond = None

    m = re.search(r"(D\d+)", s)
    dyad = m.group(1) if m else None

    return cond, dyad


def find_mp_pose_xlsx(dyad: str, cond: str, pid: str, root: Path):
    """
    Compatible avec :
    DATA/vidéo/D01/P1/SEATED/SEATEDD01_P1_pose.xlsx
    """
    candidates = []

    expected_name = f"{cond}{dyad}_{pid}_pose.xlsx".upper()

    for f in root.rglob("*_pose.xlsx"):
        p = str(f).upper().replace("\\", "/")
        fname = f.name.upper()

        has_dyad_folder = f"/{dyad}/" in p
        has_pid_folder = f"/{pid}/" in p
        has_cond_folder = f"/{cond}/" in p
        has_expected_name = fname == expected_name

        if has_dyad_folder and has_pid_folder and has_cond_folder and has_expected_name:
            candidates.append(f)

    if len(candidates) > 0:
        candidates.sort(key=lambda x: len(str(x)))
        return candidates[0]

    # recherche plus permissive si le nom exact n'est pas trouvé
    for f in root.rglob("*_pose.xlsx"):
        p = str(f).upper().replace("\\", "/")
        fname = f.name.upper()

        if dyad in p and pid in p and cond in p and fname.endswith("_POSE.XLSX"):
            candidates.append(f)

    if len(candidates) > 0:
        candidates.sort(key=lambda x: len(str(x)))
        return candidates[0]

    return None


# ============================================================
# 3) Matching frames
# ============================================================
def map_mp_to_vicon_idx(n_mp, n_vicon, fps_mp=30.0, fps_vicon=100.0):
    idx = np.rint(np.arange(n_mp) * (fps_vicon / fps_mp)).astype(int)
    idx = np.clip(idx, 0, n_vicon - 1)
    return idx


# ============================================================
# 4) Géométrie
# ============================================================
def pick_plane(arr3, plane="XZ"):
    plane = plane.upper()

    if plane == "XY":
        return arr3[:, [0, 1]]
    if plane == "XZ":
        return arr3[:, [0, 2]]
    if plane == "YZ":
        return arr3[:, [1, 2]]

    raise ValueError("plane must be XY, XZ, or YZ")


def angle_at_joint(A, B, C):
    BA = A - B
    BC = C - B

    num = np.sum(BA * BC, axis=1)
    den = np.linalg.norm(BA, axis=1) * np.linalg.norm(BC, axis=1)

    with np.errstate(divide="ignore", invalid="ignore"):
        cosang = num / den

    cosang = np.clip(cosang, -1.0, 1.0)
    ang = np.degrees(np.arccos(cosang))
    ang[~np.isfinite(ang)] = np.nan

    return ang


def summarize_series(x):
    x = x[np.isfinite(x)]

    if x.size == 0:
        return dict(mean=np.nan, sd=np.nan, min=np.nan, max=np.nan, pp=np.nan)

    return dict(
        mean=float(np.mean(x)),
        sd=float(np.std(x, ddof=1)) if x.size > 1 else 0.0,
        min=float(np.min(x)),
        max=float(np.max(x)),
        pp=float(np.max(x) - np.min(x))
    )


def corr_rmse(x, y):
    m = np.isfinite(x) & np.isfinite(y)

    if m.sum() < 3:
        return np.nan, np.nan

    xx = x[m]
    yy = y[m]

    r = float(np.corrcoef(xx, yy)[0, 1])
    rmse = float(np.sqrt(np.mean((xx - yy) ** 2)))

    return r, rmse


# ============================================================
# 5) Repère épaules 2D
# ============================================================
def _norm2(v):
    n = np.linalg.norm(v, axis=1, keepdims=True)
    n[n == 0] = np.nan
    return v / n


def build_shoulder_frame_2d(Ls, Rs):
    C = (Ls + Rs) / 2.0
    u = _norm2(Rs - Ls)
    v = np.stack([-u[:, 1], u[:, 0]], axis=1)

    return C, u, v


def project_to_frame_2d(P, C, u, v):
    Q = P - C
    x = np.sum(Q * u, axis=1)
    y = np.sum(Q * v, axis=1)

    return np.stack([x, y], axis=1)


def mp_triplet_in_shoulder_frame(df_pose, side="RIGHT"):
    side = side.upper()

    required_cols = [
        "LEFT_SHOULDER_x", "LEFT_SHOULDER_y",
        "RIGHT_SHOULDER_x", "RIGHT_SHOULDER_y",
        f"{side}_WRIST_x", f"{side}_WRIST_y",
        f"{side}_ELBOW_x", f"{side}_ELBOW_y",
        f"{side}_SHOULDER_x", f"{side}_SHOULDER_y"
    ]

    missing = [c for c in required_cols if c not in df_pose.columns]

    if missing:
        raise ValueError(f"colonnes MediaPipe manquantes : {missing}")

    Ls = df_pose[["LEFT_SHOULDER_x", "LEFT_SHOULDER_y"]].to_numpy(float)
    Rs = df_pose[["RIGHT_SHOULDER_x", "RIGHT_SHOULDER_y"]].to_numpy(float)

    C, u, v = build_shoulder_frame_2d(Ls, Rs)

    W = df_pose[[f"{side}_WRIST_x", f"{side}_WRIST_y"]].to_numpy(float)
    E = df_pose[[f"{side}_ELBOW_x", f"{side}_ELBOW_y"]].to_numpy(float)
    S = df_pose[[f"{side}_SHOULDER_x", f"{side}_SHOULDER_y"]].to_numpy(float)

    Wp = project_to_frame_2d(W, C, u, v)
    Ep = project_to_frame_2d(E, C, u, v)
    Sp = project_to_frame_2d(S, C, u, v)

    return Wp, Ep, Sp


# ============================================================
# 6) Extraction Vicon
# ============================================================
def vicon_triplet(df_v, cols, pid="P1", side="D"):
    if pid == "P1":
        wtoken = f"poignet_{side}"
        etoken = f"coude_{side}"
        stoken = f"epaule_{side}"
    else:
        wtoken = f"2poignet_{side}"
        stoken = f"2epaule_{side}"

        elbow_candidates = [
            f"2coude_{side}",
            f"2coudes_{side}",
            f"2elbow_{side}",
            f"elbow2_{side}"
        ]

        etoken = None

        for cand in elbow_candidates:
            x, y, z = find_xyz_cols(cols, cand)

            if None not in [x, y, z]:
                etoken = cand
                break

        if etoken is None:
            etoken = f"2coude_{side}"

    wX, wY, wZ = find_xyz_cols(cols, wtoken)
    eX, eY, eZ = find_xyz_cols(cols, etoken)
    sX, sY, sZ = find_xyz_cols(cols, stoken)

    if None in [wX, wY, wZ, eX, eY, eZ, sX, sY, sZ]:
        return None, dict(
            tokens=(wtoken, etoken, stoken),
            missing=dict(
                wrist=(wX, wY, wZ),
                elbow=(eX, eY, eZ),
                shoulder=(sX, sY, sZ)
            )
        )

    W = df_v[[wX, wY, wZ]].to_numpy(float)
    E = df_v[[eX, eY, eZ]].to_numpy(float)
    S = df_v[[sX, sY, sZ]].to_numpy(float)

    return (W, E, S), dict(tokens=(wtoken, etoken, stoken))


def vicon_shoulders_3d(df_v, cols, pid="P1"):
    if pid == "P1":
        tok_L = "epaule_G"
        tok_R = "epaule_D"
    else:
        tok_L = "2epaule_G"
        tok_R = "2epaule_D"

    LX, LY, LZ = find_xyz_cols(cols, tok_L)
    RX, RY, RZ = find_xyz_cols(cols, tok_R)

    if None in [LX, LY, LZ, RX, RY, RZ]:
        return None

    L = df_v[[LX, LY, LZ]].to_numpy(float)
    R = df_v[[RX, RY, RZ]].to_numpy(float)

    return L, R


def vicon_triplet_in_shoulder_frame_2d(df_v, cols, pid="P1", side="D", plane="XZ"):
    trip, info = vicon_triplet(df_v, cols, pid=pid, side=side)

    if trip is None:
        return None

    W3, E3, S3 = trip

    sh = vicon_shoulders_3d(df_v, cols, pid=pid)

    if sh is None:
        return None

    L3, R3 = sh

    W2 = pick_plane(W3, plane)
    E2 = pick_plane(E3, plane)
    S2 = pick_plane(S3, plane)
    L2 = pick_plane(L3, plane)
    R2 = pick_plane(R3, plane)

    C, u, v = build_shoulder_frame_2d(L2, R2)

    Wp = project_to_frame_2d(W2, C, u, v)
    Ep = project_to_frame_2d(E2, C, u, v)
    Sp = project_to_frame_2d(S2, C, u, v)

    return Wp, Ep, Sp


# ============================================================
# 7) Batch
# ============================================================
rows = []

for csv_path in sorted(ROOT_VICON.rglob("*.csv")):
    cond, dyad = parse_vicon_name(csv_path.name)

    if cond is None or dyad is None:
        continue

    df_v = read_vicon_csv(csv_path)
    cols = list(df_v.columns)
    n_v = len(df_v)

    mp_p1 = find_mp_pose_xlsx(dyad, cond, "P1", ROOT_MP)
    mp_p2 = find_mp_pose_xlsx(dyad, cond, "P2", ROOT_MP)

    base = dict(
        file=csv_path.name,
        vicon_csv=str(csv_path),
        dyad=dyad,
        condition=cond,
        mp_pose_p1=str(mp_p1) if mp_p1 else None,
        mp_pose_p2=str(mp_p2) if mp_p2 else None,
        plane_2D=PLANE_2D
    )

    for pid, mp_path in [("P1", mp_p1), ("P2", mp_p2)]:
        out = base.copy()
        out["pid"] = pid
        out["MP_plane_used"] = "XY"

        if mp_path is None:
            out["error"] = "missing mp pose.xlsx"
            rows.append(out)
            continue

        try:
            df_mp = pd.read_excel(mp_path)
            n_mp = len(df_mp)
            idx_v = map_mp_to_vicon_idx(n_mp, n_v, FPS_MP, FPS_VICON)

            WR, ER, SR = mp_triplet_in_shoulder_frame(df_mp, "RIGHT")
            WL, EL, SL = mp_triplet_in_shoulder_frame(df_mp, "LEFT")

            ang_mp_R = angle_at_joint(SR, ER, WR)
            ang_mp_L = angle_at_joint(SL, EL, WL)

            smpR = summarize_series(ang_mp_R)
            smpL = summarize_series(ang_mp_L)

            out.update({f"MP_R_{k}": v for k, v in smpR.items()})
            out.update({f"MP_L_{k}": v for k, v in smpL.items()})
            out["MP_mean_of_means"] = np.nanmean([out["MP_R_mean"], out["MP_L_mean"]])

            tripR, infoR = vicon_triplet(df_v, cols, pid=pid, side="D")
            tripL, infoL = vicon_triplet(df_v, cols, pid=pid, side="G")

            if tripR is None or tripL is None:
                out["error"] = f"missing vicon cols. R={infoR} L={infoL}"
                rows.append(out)
                continue

            W3R, E3R, S3R = tripR
            W3L, E3L, S3L = tripL

            W3R = W3R[idx_v]
            E3R = E3R[idx_v]
            S3R = S3R[idx_v]

            W3L = W3L[idx_v]
            E3L = E3L[idx_v]
            S3L = S3L[idx_v]

            ang_vi3_R = angle_at_joint(S3R, E3R, W3R)
            ang_vi3_L = angle_at_joint(S3L, E3L, W3L)

            svi3R = summarize_series(ang_vi3_R)
            svi3L = summarize_series(ang_vi3_L)

            out.update({f"VI3_R_{k}": v for k, v in svi3R.items()})
            out.update({f"VI3_L_{k}": v for k, v in svi3L.items()})
            out["VI3_mean_of_means"] = np.nanmean([out["VI3_R_mean"], out["VI3_L_mean"]])

            VI2R = vicon_triplet_in_shoulder_frame_2d(df_v, cols, pid=pid, side="D", plane=PLANE_2D)
            VI2L = vicon_triplet_in_shoulder_frame_2d(df_v, cols, pid=pid, side="G", plane=PLANE_2D)

            if VI2R is None or VI2L is None:
                out["error"] = "missing vicon shoulders or joint cols for body-frame 2D"
                rows.append(out)
                continue

            W2R, E2R, S2R = VI2R
            W2L, E2L, S2L = VI2L

            W2R = W2R[idx_v]
            E2R = E2R[idx_v]
            S2R = S2R[idx_v]

            W2L = W2L[idx_v]
            E2L = E2L[idx_v]
            S2L = S2L[idx_v]

            ang_vi2_R = angle_at_joint(S2R, E2R, W2R)
            ang_vi2_L = angle_at_joint(S2L, E2L, W2L)

            svi2R = summarize_series(ang_vi2_R)
            svi2L = summarize_series(ang_vi2_L)

            out.update({f"VI2_R_{k}": v for k, v in svi2R.items()})
            out.update({f"VI2_L_{k}": v for k, v in svi2L.items()})
            out["VI2_mean_of_means"] = np.nanmean([out["VI2_R_mean"], out["VI2_L_mean"]])

            rR_2d, rmseR_2d = corr_rmse(ang_mp_R, ang_vi2_R)
            rL_2d, rmseL_2d = corr_rmse(ang_mp_L, ang_vi2_L)

            out["corr_MP_vs_VI2_R"] = rR_2d
            out["rmse_MP_vs_VI2_R"] = rmseR_2d
            out["corr_MP_vs_VI2_L"] = rL_2d
            out["rmse_MP_vs_VI2_L"] = rmseL_2d

            rR_3d, rmseR_3d = corr_rmse(ang_mp_R, ang_vi3_R)
            rL_3d, rmseL_3d = corr_rmse(ang_mp_L, ang_vi3_L)

            out["corr_MP_vs_VI3_R"] = rR_3d
            out["rmse_MP_vs_VI3_R"] = rmseR_3d
            out["corr_MP_vs_VI3_L"] = rL_3d
            out["rmse_MP_vs_VI3_L"] = rmseL_3d

            out["error"] = ""

        except Exception as e:
            out["error"] = str(e)

        rows.append(out)


df_out = pd.DataFrame(rows)

print("Nombre de lignes créées =", len(df_out))
print(df_out["error"].value_counts(dropna=False).head(20))

df_out.to_excel(OUT_PATH, index=False)

print("✅ Saved:", OUT_PATH)
print(df_out.head())

PROJECT_ROOT = /Users/matysprecloux/Desktop/SYNCOGEST
DATA_DIR = /Users/matysprecloux/Desktop/SYNCOGEST/DATA
ROOT_VICON = /Users/matysprecloux/Desktop/SYNCOGEST/DATA/VICON_CSV True
ROOT_MP = /Users/matysprecloux/Desktop/SYNCOGEST/DATA/video True
Nb CSV Vicon = 60
Nb pose MediaPipe = 120
Nombre de lignes créées = 120
error
    120
Name: count, dtype: int64
✅ Saved: /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code/ANGLE_elbow_MP_vs_VICON_planeYZ_noInterp.xlsx
            file                                          vicon_csv dyad  \
0  SEATEDD01.csv  /Users/matysprecloux/Desktop/SYNCOGEST/DATA/VI...  D01   
1  SEATEDD01.csv  /Users/matysprecloux/Desktop/SYNCOGEST/DATA/VI...  D01   
2  SEATEDD02.csv  /Users/matysprecloux/Desktop/SYNCOGEST/DATA/VI...  D02   
3  SEATEDD02.csv  /Users/matysprecloux/Desktop/SYNCOGEST/DATA/VI...  D02   
4  SEATEDD03.csv  /Users/matysprecloux/Desktop/SYNCOGEST/DATA/VI...  D03   

  condition                                         mp_pose_p1  \
0    SEA